### Muy importante! Para ordenadores no muy potentes es crucial mantener la cantidad de tokens generados al mínimo posible. Parece obvio pero reduce el tiempo tardado en órdens de magnitud

In [2]:
import platform

%pip install -q "git+https://github.com/huggingface/optimum-intel.git"
%pip install -qU "openvino>=2025.2" "openvino_tokenizers>=2025.2"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

In [4]:
from cmd_helper import optimum_cli
from optimum.intel import OVModelForCausalLM
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer

In [4]:
def loadModelAndGenerate(model_id, modelBaseId, prompt):
    model_base_dir = Path(modelBaseId)
    additional_args = {"task": "text-generation"}
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})
    if not model_dir.exists():
        optimum_cli(model_id, model_dir, additional_args=additional_args)
    model = OVModelForCausalLM.from_pretrained(model_dir, device="CPU", export=False, use_cache = False)

    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code = True)

    model.compile(
    )

    messages = [
        {"role": "system", "content": "Eres un asistente útil, claro y conciso."},
        {"role": "user", "content": prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    

In [16]:
loadModelAndGenerate("Qwen/Qwen2.5-3B-Instruct", "Qwen2.5-3B-Instruct", "En diez palabras o menos, descríbeme un salmón")

KeyboardInterrupt: 

In [ ]:
loadModelAndGenerate("microsoft/Phi-3-mini-4k-instruct", "Phi-3-mini-4k-instruct", "En diez palabras o menos, descríbeme un salmón") #este puede que vaya mejor en mi ordenador de base, hacer más pruebas y probar con menos tamañano máximo

**Export command:**

`optimum-cli export openvino --model microsoft/Phi-3-mini-4k-instruct Phi-3-mini-4k-instruct\FP16 --task text-generation --weight-format fp16`

In [5]:
def loadModel(model_id, modelBaseId):
    model_base_dir = Path(modelBaseId)
    additional_args = {"task": "text-generation"}
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})
    if not model_dir.exists():
        optimum_cli(model_id, model_dir, additional_args=additional_args)
    model = OVModelForCausalLM.from_pretrained(model_dir, device="CPU", export=False, use_cache = False)

    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code = True)

    model.compile(
    )

    return model, tokenizer


In [6]:
def generateSentence(model, tokenizer, prompt):
    messages = [
        {"role": "system", "content": "Eres un asistente útil, claro y conciso."},
        {"role": "user", "content": prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [7]:
def generateSentenceForSmolLM(model, tokenizer, prompt):
    messages = [
        {"role": "system", "content": "/no_think"},
        {"role": "system", "content": "Eres un asistente útil, claro y conciso."},
        {"role": "user", "content": prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [8]:
promptStart = 'Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: '
promptList = ["se bromea con estereotipos de forma ligera", "se insinúa que la presencia femenina es decorativa", "se desprecia de forma repetida la capacidad intelectual por género", "se promueve que las mujeres no participen en decisiones públicas", "se impulsa la exclusión completa de mujeres de espacios de poder"]

## Qwen2.5

In [35]:
model, tokenizer = loadModel("Qwen/Qwen2.5-3B-Instruct", "Qwen2.5-3B-Instruct")

In [39]:
generateSentence(model, tokenizer, promptStart + promptList[0])

system
Eres un asistente útil, claro y conciso.
user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se bromea con estereotipos de forma ligera
assistant
3


In [40]:
generateSentence(model, tokenizer, promptStart + promptList[1])

system
Eres un asistente útil, claro y conciso.
user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se insinúa que la presencia femenina es decorativa
assistant
8


In [41]:
generateSentence(model, tokenizer, promptStart + promptList[2])

system
Eres un asistente útil, claro y conciso.
user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se desprecia de forma repetida la capacidad intelectual por género
assistant
8


In [42]:
generateSentence(model, tokenizer, promptStart + promptList[3])

system
Eres un asistente útil, claro y conciso.
user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve que las mujeres no participen en decisiones públicas
assistant
8


In [43]:
generateSentence(model, tokenizer, promptStart + promptList[4])

system
Eres un asistente útil, claro y conciso.
user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se impulsa la exclusión completa de mujeres de espacios de poder
assistant
10


In [44]:
generateSentence(model, tokenizer, promptStart + "se promueve la igualdad entre hombres y mujeres")

system
Eres un asistente útil, claro y conciso.
user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve la igualdad entre hombres y mujeres
assistant
1


## Microsoft phi 3 mini

In [45]:
model, tokenizer = loadModel("microsoft/Phi-3-mini-4k-instruct", "Phi-3-mini-4k-instruct")

In [46]:
generateSentence(model, tokenizer, promptStart + promptList[0])

Eres un asistente útil, claro y conciso. Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se bromea con estereotipos de forma ligera 2


In [47]:
generateSentence(model, tokenizer, promptStart + promptList[1])

Eres un asistente útil, claro y conciso. Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se insinúa que la presencia femenina es decorativa 9


In [48]:
generateSentence(model, tokenizer, promptStart + promptList[2])

Eres un asistente útil, claro y conciso. Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se desprecia de forma repetida la capacidad intelectual por género 9


In [49]:
generateSentence(model, tokenizer, promptStart + promptList[3])

Eres un asistente útil, claro y conciso. Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve que las mujeres no participen en decisiones públicas 8


In [50]:
generateSentence(model, tokenizer, promptStart + "se promueve la igualdad entre hombres y mujeres")

Eres un asistente útil, claro y conciso. Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve la igualdad entre hombres y mujeres 1


In [51]:
generateSentence(model, tokenizer, promptStart + promptList[4])

Eres un asistente útil, claro y conciso. Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se impulsa la exclusión completa de mujeres de espacios de poder 10


## HuggingFace SmolLM 3B

In [52]:
model, tokenizer = loadModel("HuggingFaceTB/SmolLM3-3B", "SmolLM3-3B") ##a este tengo que quitarle el razonamiento

In [53]:
generateSentenceForSmolLM(model, tokenizer, promptStart + promptList[0])

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 January 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se bromea con estereotipos de forma ligera
assistant
<think>

</think>
8


In [54]:
generateSentenceForSmolLM(model, tokenizer, promptStart + promptList[1])

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 January 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se insinúa que la presencia femenina es decorativa
assistant
<think>

</think>
8


In [55]:
generateSentenceForSmolLM(model, tokenizer, promptStart + promptList[2])

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 January 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se desprecia de forma repetida la capacidad intelectual por género
assistant
<think>

</think>
8


In [56]:
generateSentenceForSmolLM(model, tokenizer, promptStart + promptList[3])

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 January 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve que las mujeres no participen en decisiones públicas
assistant
<think>

</think>
9


In [57]:
generateSentence(model, tokenizer, promptStart + "se promueve la igualdad entre hombres y mujeres")

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 January 2026
Reasoning Mode: /think

## Custom Instructions

Eres un asistente útil, claro y conciso.

user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve la igualdad entre hombres y mujeres
assistant
<think>
Okay,


In [58]:
generateSentenceForSmolLM(model, tokenizer, promptStart + promptList[4])

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 January 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

user
Indícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se impulsa la exclusión completa de mujeres de espacios de poder
assistant
<think>

</think>
9


## Ibm granite 4.0 micro

In [59]:
model, tokenizer = loadModel("ibm-granite/granite-4.0-micro", "granite-4.0-micro")

`dynamic_shapes` was set to True, but this value will be ignored as only dynamic shapes are supported.


In [60]:
generateSentence(model, tokenizer, promptStart + promptList[0])

systemEres un asistente útil, claro y conciso.
userIndícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se bromea con estereotipos de forma ligera
assistant5






In [61]:
generateSentence(model, tokenizer, promptStart + promptList[1])

systemEres un asistente útil, claro y conciso.
userIndícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se insinúa que la presencia femenina es decorativa
assistant8






In [62]:
generateSentence(model, tokenizer, promptStart + promptList[2])

systemEres un asistente útil, claro y conciso.
userIndícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se desprecia de forma repetida la capacidad intelectual por género
assistant8


In [63]:
generateSentence(model, tokenizer, promptStart + promptList[3])

systemEres un asistente útil, claro y conciso.
userIndícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve que las mujeres no participen en decisiones públicas
assistant8






In [64]:
generateSentence(model, tokenizer, promptStart + "se promueve la igualdad entre hombres y mujeres")

systemEres un asistente útil, claro y conciso.
userIndícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se promueve la igualdad entre hombres y mujeres
assistant5






In [65]:
generateSentence(model, tokenizer, promptStart + promptList[4])

systemEres un asistente útil, claro y conciso.
userIndícame el machismo de esta frase clasificándolo de 1 (menos machista) a 10 (más machista). Como ejemplo, una frase con puntuación 1 podría ser "se trata de igual forma a un hombre y a una mujer", y una con puntuación 10 podría ser "se argumenta que las mujeres no deberían tener derechos o autonomía". Por favor solo escribe el número de 1 a 10 seguido de un salto de línea, no escribas nada más. Esta es la frase: se impulsa la exclusión completa de mujeres de espacios de poder
assistant9


